<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento Digital de Imágenes*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

# 1. Modelos de Difusión (Diffusion Models) - Fundamentos Teóricos

Los Modelos de Difusión Probabilística Generativa (DDPM) representan el estándar actual para la generación de imágenes de alta fidelidad. A diferencia de las GANs, que utilizan un juego minimax basado en dos redes que compiten, o los VAEs, que proyectan los datos a un espacio latente aproximado en un solo paso, los modelos de difusión generan datos aprendiendo a **revertir un proceso sistemático de destrucción de información**.

El principio fundamental se divide en dos fases matemáticas discretas distribuidas a lo largo de un proceso de tiempo de Markov con $T$ pasos (típicamente $T = 1000$).

---

## 1.1. El Proceso de Difusión Hacia Adelante (Forward Process / $q$)

El proceso hacia adelante es determinista y no requiere entrenamiento. Consiste en tomar una imagen real de la distribución de datos, $x_0 \sim q(x)$, y agregarle ruido gaussiano de forma iterativa y controlada a lo largo de $T$ pasos, siguiendo un programa de varianza (*variance schedule*) predefinido $\beta_1, \beta_2, \dots, \beta_T$.

En cada paso individual $t$, la transición condicional se define matemáticamente como:
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t}x_{t-1}, \beta_t\mathbf{I})$$

### El Truco del "Paso Directo" (Trick for Direct Sampling)
Calcular este proceso paso por paso para llegar a un tiempo $t$ avanzado sería computacionalmente ineficiente. Definiendo $\alpha_t = 1 - \beta_t$ y la barra acumulada $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$, podemos derivar una propiedad matemática que permite muestrear el tensor en cualquier paso $t$ directamente desde la imagen original $x_0$:

$$x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1 - \bar{\alpha}_t}\epsilon \quad \text{donde} \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})$$

*   **Modificación del Shape del Tensor:** A lo largo del proceso hacia adelante, las dimensiones del tensor se mantienen estrictamente invariantes en su forma espacial y de canales:
    $$\text{Shape del Tensor} = (B, C, H, W)$$
    Lo que se modifica de manera paulatina es la distribución estadística interna de sus valores. Cuando $t \to T$, el coeficiente $\sqrt{\bar{\alpha}_T} \to 0$, provocando que la estructura de la imagen original colapse por completo, transformando el tensor en ruido blanco gaussiano puro.

---

## 1.2. El Proceso de Difusión Inversa (Reverse Process / $p_\theta$)

El verdadero núcleo de la inteligencia artificial generativa en estos modelos radica en el proceso inverso. Si conocemos la transición inversa exacta $q(x_{t-1} | x_t)$, podríamos tomar un bloque de ruido aleatorio $x_T \sim \mathcal{N}(0, \mathbf{I})$ y remover el ruido paso a paso hasta recuperar una imagen limpia. 

Como $q(x_{t-1} | x_t)$ depende de la distribución de toda la base de datos (lo cual es intratable), aproximamos esta transición utilizando una red neuronal parametrizada por pesos $\theta$:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))$$

### Arquitectura y Predicción del Ruido
En la práctica, en lugar de predecir directamente la media del tensor limpio $\mu_\theta$, se entrena a la red neuronal (comúnmente una arquitectura **U-Net** modificada con capas de atención e inyección de embeddings de tiempo $t$) para que prediga el **ruido exacto $\epsilon$** que fue añadido en ese instante específico.

*   **Flujo de Inferencia:** En cada paso descendente desde $t$ hasta $1$, el modelo toma el tensor ruidoso actual $x_t$, estima el ruido latente con la red $\epsilon_\theta(x_t, t)$ y calcula el estado anterior mediante la ecuación de muestreo:
    $$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right) + \sigma_t z \quad \text{donde} \quad z \sim \mathcal{N}(0, \mathbf{I})$$

A través de este desvanecimiento iterativo del ruido estimado, los patrones geométricos emergen orgánicamente del caos probabilístico, alcanzando el tensor final de imagen limpia $x_0 \in \mathbb{R}^{B \times C \times H \times W}$.

# 2. Implementación de código

In [ ]:
%%writefile ddpm_ddp.py
import os
import math
from tqdm import tqdm
import matplotlib.pyplot as plt
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.amp import GradScaler, autocast
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms

from torchmetrics.image.fid import FrechetInceptionDistance

# Corregir importación faltante del scheduler
from torch.optim.lr_scheduler import CosineAnnealingLR

# ===================================================================
# 1. Funciones de Configuración DDP
# ===================================================================
def setup_ddp():
    """
    Inicializa el grupo de procesos. Asume que el script se lanzará con torchrun.
    """
    dist.init_process_group(backend="nccl") # NCCL es el backend optimizado para GPUs NVIDIA
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    return local_rank

def cleanup_ddp():
    """Limpia el grupo de procesos al terminar."""
    dist.destroy_process_group()

# ===================================================================
# 2. Dataloader adaptado para DDP (MNIST - Canales: 1, Tamaño: 28x28)
# ===================================================================
def get_mnist_dataloaders_ddp(
    data_dir: str = './data',
    batch_size: int = 128,
    num_workers: int = 4
) -> Tuple[DataLoader, DataLoader, DistributedSampler]:
    """
    Prepara los DataLoaders usando DistributedSampler adaptados para MNIST.
    Nota pedagógica: MNIST viene en un formato espacial de 28x28 píxeles. 
    Para evitar romper la simetría de divisiones sucesivas entre 2 del cuello de botella 
    de la U-Net (28 -> 14 -> 7 -> ¿?), aplicamos un padding simétrico para llevarlo a 32x32.
    """
    transform = transforms.Compose([
        transforms.Pad(2), # Añade 2 píxeles por lado: Tensor de (1, 28, 28) -> (1, 32, 32)
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5], std=[0.5]) # Un solo canal en escala de grises
    ])

    train_dataset = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)

    # El sampler se encarga de dividir el dataset entre las GPUs
    train_sampler = DistributedSampler(train_dataset)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler, # IMPORTANTE: Al usar sampler, NO se puede usar shuffle=True
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )

    # Generalmente la validación/test se hace en una sola GPU (rank 0) para simplificar
    # el cálculo de métricas globales como el FID, por lo que no distribuimos este sampler.
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    return train_loader, test_loader, train_sampler

# ===================================================================
# 3. Definición del modelo UNet para DDPM
# ===================================================================
class SinusoidalPositionEmbeddings(nn.Module):
    """
    Codificación de la variable de tiempo t usando embeddings sinusoidales.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class Block(nn.Module):
    """Bloque base con Conv2d, GroupNorm y activación SiLU (Swish)."""
    def __init__(self, in_channels, out_channels, groups=32):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.norm = nn.GroupNorm(groups, out_channels)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.norm(self.proj(x)))

class ResnetBlock(nn.Module):
    """
    Bloque residual que integra la información del tiempo t.
    El embedding de tiempo se proyecta y se suma a los feature maps.
    """
    def __init__(self, in_channels, out_channels, time_emb_dim, groups=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        
        self.block1 = Block(in_channels, out_channels, groups=groups)
        self.block2 = Block(out_channels, out_channels, groups=groups)
        
        # Conexión residual (shortcut) en caso de cambio de dimensiones
        self.res_conv = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, time_emb):
        h = self.block1(x)
        
        # Inyección del embedding de tiempo
        time_hidden = self.mlp(time_emb)
        h = h + time_hidden[:, :, None, None]
        
        h = self.block2(h)
        return h + self.res_conv(x)

class AttentionBlock(nn.Module):
    """
    Mecanismo de Self-Attention estándar aplicado espacialmente.
    El paper lo utiliza en la resolución de 16x16.
    """
    def __init__(self, channels, groups=32):
        super().__init__()
        self.norm = nn.GroupNorm(groups, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        
        qkv = self.qkv(h).view(B, 3, C, H * W)
        q, k, v = qkv.unbind(dim=1)
        
        # Atención escalar por producto punto (Dot-product attention)
        attn = torch.einsum("b c i, b c j -> b i j", q, k) * (int(C) ** (-0.5))
        attn = F.softmax(attn, dim=-1)
        
        out = torch.einsum("b i j, b c j -> b c i", attn, v)
        out = out.view(B, C, H, W)
        
        return self.proj(out) + x

class DDPM_UNet(nn.Module):
    """
    Arquitectura U-Net principal para el proceso inverso del DDPM.
    Modificada con soporte explícito de canales para MNIST (in_channels=1, out_channels=1).
    """
    def __init__(self, in_channels=1, out_channels=1, base_channels=64, time_emb_dim=256):
        super().__init__()
        
        # Procesamiento del tiempo
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim)
        )
        
        self.init_conv = nn.Conv2d(in_channels, base_channels, kernel_size=3, padding=1)
        
        # Encoder (Downsampling)
        # 32x32
        self.down1 = ResnetBlock(base_channels, base_channels, time_emb_dim)
        self.down1_pool = nn.MaxPool2d(2)
        
        # 16x16 - Aquí aplicamos Self-Attention según el paper
        self.down2 = ResnetBlock(base_channels, base_channels * 2, time_emb_dim)
        self.attn_down = AttentionBlock(base_channels * 2)
        self.down2_pool = nn.MaxPool2d(2)
        
        # 8x8
        self.down3 = ResnetBlock(base_channels * 2, base_channels * 4, time_emb_dim)
        self.down3_pool = nn.MaxPool2d(2)
        
        # Bottleneck (4x4)
        self.mid_block1 = ResnetBlock(base_channels * 4, base_channels * 4, time_emb_dim)
        self.mid_attn = AttentionBlock(base_channels * 4)
        self.mid_block2 = ResnetBlock(base_channels * 4, base_channels * 4, time_emb_dim)
        
        # Decoder (Upsampling)
        # Las entradas al decoder son el doble de canales debido a la concatenación (skip connections)
        
        # 4x4 -> 8x8
        self.up1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.up_res1 = ResnetBlock(base_channels * 8, base_channels * 2, time_emb_dim)
        
        # 8x8 -> 16x16 - Aplicamos Self-Attention
        self.up2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.up_res2 = ResnetBlock(base_channels * 4, base_channels, time_emb_dim)
        self.attn_up = AttentionBlock(base_channels)
        
        # 16x16 -> 32x32
        self.up3 = nn.Upsample(scale_factor=2, mode='nearest')
        self.up_res3 = ResnetBlock(base_channels * 2, base_channels, time_emb_dim)
        
        # Salida final
        self.final_block = Block(base_channels, base_channels)
        self.final_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x, time):
        # 1. Embedding del tiempo
        t_emb = self.time_mlp(time)
        
        # 2. Encoder
        x0 = self.init_conv(x)
        x1 = self.down1(x0, t_emb)
        
        x2 = self.down2(self.down1_pool(x1), t_emb)
        x2 = self.attn_down(x2) # Atención en 16x16
        
        x3 = self.down3(self.down2_pool(x2), t_emb)
        x4 = self.down3_pool(x3)
        
        # 3. Bottleneck
        mid = self.mid_block1(x4, t_emb)
        mid = self.mid_attn(mid)
        mid = self.mid_block2(mid, t_emb)
        
        # 4. Decoder con Skip Connections
        u1 = self.up1(mid)
        u1 = torch.cat([u1, x3], dim=1) # Concatenación de features
        u1 = self.up_res1(u1, t_emb)
        
        u2 = self.up2(u1)
        u2 = torch.cat([u2, x2], dim=1)
        u2 = self.up_res2(u2, t_emb)
        u2 = self.attn_up(u2) # Atención en 16x16
        
        u3 = self.up3(u2)
        u3 = torch.cat([u3, x1], dim=1)
        u3 = self.up_res3(u3, t_emb)
        
        # 5. Salida
        out = self.final_block(u3)
        out = self.final_conv(out)
        
        return out

# ===================================================================
# 4. Funciones auxiliares
# ===================================================================
class GaussianDiffusion:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = T
        self.device = device
        
        # Schedule de varianza (betas)
        self.betas = torch.linspace(beta_start, beta_end, T).to(device)
        self.alphas = 1.0 - self.betas
        
        # alpha_hat: producto acumulado de alphas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        
    def q_sample(self, x_0, t, noise=None):
        """
        Forward process: Ecuación 4. Muestrea x_t dado x_0 de forma cerrada.
        """
        if noise is None:
            noise = torch.randn_like(x_0).to(self.device)
            
        alpha_hat_t = self.alphas_cumprod[t].view(-1, 1, 1, 1)
        
        # x_t = sqrt(alpha_hat) * x_0 + sqrt(1 - alpha_hat) * epsilon
        x_t = torch.sqrt(alpha_hat_t) * x_0 + torch.sqrt(1.0 - alpha_hat_t) * noise
        return x_t, noise

@torch.no_grad()
def generate_and_plot_sample(model, diffusion, device, epoch, experiment_name, img_size=32):
    """
    Implementa el Algoritmo 2 (Sampling) para generar una imagen partiendo de ruido puro.
    Adaptado a 1 canal (Escala de grises - MNIST).
    """
    model.eval()
    
    # x_T ~ N(0, I) - Cambiado a 1 canal de entrada
    x = torch.randn((1, 1, img_size, img_size), device=device)
    
    # Bucle reverso: de T-1 hasta 0
    for i in reversed(range(diffusion.T)):
        t_batch = torch.tensor([i], device=device).long()
        
        # Predecir epsilon_theta
        predicted_noise = model(x, t_batch)
        
        # Variables estadísticas fijadas por el scheduler
        alpha_t = diffusion.alphas[i]
        alpha_hat_t = diffusion.alphas_cumprod[i]
        beta_t = diffusion.betas[i]
        
        # Ecuación del paso de limpieza (Algoritmo 2)
        coeff = 1.0 / torch.sqrt(alpha_t)
        noise_weight = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_hat_t)
        
        mean = coeff * (x - noise_weight * predicted_noise)
        
        if i > 0:
            noise = torch.randn_like(x)
            sigma_t = torch.sqrt(beta_t) # sigma_t^2 = beta_t
            x = mean + sigma_t * noise
        else:
            x = mean # No agregar ruido en el último paso (t=0)
            
    # Transformar el tensor [-1, 1] a formato estándar [0, 1] para escala de grises
    x = (x.clamp(-1, 1) + 1) / 2
    img = x[0, 0].cpu().numpy() # Extraer único canal

    os.makedirs(experiment_name, exist_ok=True)
    os.makedirs(f"{experiment_name}/plot_per_epoch", exist_ok=True)
    
    plt.figure(figsize=(3, 3))
    plt.imshow(img, cmap="gray") # Mostrar como mapa de grises nativo
    plt.title(f"Muestra generada - Epoch {epoch}")
    plt.axis("off")
    plt.savefig(f"{experiment_name}/plot_per_epoch/epoch_{epoch}.png")
    plt.close()

@torch.no_grad()
def calculate_fid(
    model: torch.nn.Module,
    diffusion, # Instancia de la clase GaussianDiffusion
    dataloader: torch.utils.data.DataLoader,
    device: str,
    num_samples: int = 10000, 
    batch_size: int = 128
) -> float:
    """
    Calcula el Fréchet Inception Distance (FID) del modelo.
    Nota crítica: Inception V3 espera estrictamente imágenes de 3 canales (RGB).
    Adaptamos la función repitiendo el canal de escala de grises para simular canales RGB.
    """
    model.eval()
    model.to(device)
    
    # Inception V3 espera imágenes con formato RGB y valores en [0, 255]
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    
    # ---------------------------------------------------------
    # 1. Extraer características de la distribución REAL
    # ---------------------------------------------------------
    print(f"--- Extrayendo características de {num_samples} imágenes reales ---")
    real_count = 0
    for images, _ in dataloader:
        if real_count >= num_samples:
            break
        
        # Las imágenes del dataloader están en [-1.0, 1.0] (1 canal)
        images = images.to(device)
        
        # Duplicar el canal único a 3 canales idénticos (B, 1, 32, 32) -> (B, 3, 32, 32)
        if images.shape[1] == 1:
            images = images.repeat(1, 3, 1, 1)
            
        images_uint8 = ((images + 1.0) * 127.5).clamp(0, 255).to(torch.uint8)
        
        fid.update(images_uint8, real=True)
        real_count += images.shape[0]

    # ---------------------------------------------------------
    # 2. Generar y extraer características de la distribución GENERADA
    # ---------------------------------------------------------
    print(f"--- Generando {num_samples} imágenes sintéticas ---")
    fake_count = 0
    pbar = tqdm(total=num_samples, desc="Generación (Reverse Process)")
    
    while fake_count < num_samples:
        current_batch = min(batch_size, num_samples - fake_count)
        
        # x_T ~ N(0, I) - Iniciamos con 1 canal para MNIST
        x = torch.randn((current_batch, 1, 32, 32), device=device)
        
        # Proceso de denoise paso a paso (T hasta 0)
        for i in reversed(range(diffusion.T)):
            t_batch = torch.full((current_batch,), i, device=device, dtype=torch.long)
            
            predicted_noise = model(x, t_batch)
            
            alpha_t = diffusion.alphas[i]
            alpha_hat_t = diffusion.alphas_cumprod[i]
            beta_t = diffusion.betas[i]
            
            coeff = 1.0 / torch.sqrt(alpha_t)
            noise_weight = (1.0 - alpha_t) / torch.sqrt(1.0 - alpha_hat_t)
            
            mean = coeff * (x - noise_weight * predicted_noise)
            
            if i > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(beta_t)
                x = mean + sigma_t * noise
            else:
                x = mean
                
        # Transformar las muestras generadas a un tensor de 3 canales para Inception
        x_rgb = x.repeat(1, 3, 1, 1)
        fake_images_uint8 = ((x_rgb + 1.0) * 127.5).clamp(0, 255).to(torch.uint8)
        
        fid.update(fake_images_uint8, real=False)
        
        fake_count += current_batch
        pbar.update(current_batch)
        
    pbar.close()
    
    # ---------------------------------------------------------
    # 3. Calcular la distancia de Fréchet
    # ---------------------------------------------------------
    print("--- Calculando score FID final ---")
    fid_score = fid.compute()
    fid.reset() 
    
    return fid_score.item()

# ===================================================================
# 5. Entrenamiento con DDP
# ===================================================================
def train_diffusion_model_ddp(
    model: nn.Module,
    dataloader: DataLoader,
    sampler: DistributedSampler,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler._LRScheduler,
    diffusion: GaussianDiffusion,
    epochs: int,
    local_rank: int,
    experiment_name: str
):
    # Envolver el modelo en DDP
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)

    scaler = GradScaler()
    
    for epoch in range(1, epochs + 1):
        # Garantiza un barajado distinto por época mezclando las particiones aleatorias
        sampler.set_epoch(epoch) 
        
        model.train()
        total_loss = 0.0
        
        # Solo mostrar la barra de progreso en la GPU principal (rank 0)
        if local_rank == 0:
            pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{epochs}")
        else:
            pbar = dataloader

        for images, _ in pbar:
            images = images.to(local_rank)
            B = images.shape[0]

            t = torch.randint(0, diffusion.T, (B,), device=local_rank).long()
            
            noise = torch.randn_like(images).to(local_rank)
            x_t, _ = diffusion.q_sample(images, t, noise=noise)

            optimizer.zero_grad()
            with autocast(device_type="cuda"):
                predicted_noise = model(x_t, t)
                loss = F.mse_loss(predicted_noise, noise)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Sincronizar el loss entre todas las GPUs para tener el valor global real
            loss_tensor = loss.clone()
            dist.all_reduce(loss_tensor, op=dist.ReduceOp.SUM)
            global_loss = loss_tensor.item() / dist.get_world_size()

            total_loss += global_loss
            
            if local_rank == 0:
                pbar.set_postfix({"Loss Global": f"{global_loss:.4f}"})

        scheduler.step()

        # Acciones al final de la época (Solo en Rank 0)
        if local_rank == 0:
            avg_loss = total_loss / len(dataloader)
            print(f"-> Epoch {epoch} completada. Loss promedio global: {avg_loss:.4f}\n")
            
            # Pasamos model.module para acceder a la U-Net subyacente sin el wrapper DDP
            generate_and_plot_sample(model.module, diffusion, local_rank, epoch, experiment_name)

    return model

# ===================================================================
# 6. Bloque Principal de Ejecución (Main)
# ===================================================================
if __name__ == "__main__":
    # 0. Creación del experimento
    experiment_name = "ddpm_mnist"
    os.makedirs(experiment_name, exist_ok=True)
    os.makedirs(experiment_name+"/plot_per_epoch", exist_ok=True)
    
    # 1. Inicializar el entorno distribuido
    local_rank = setup_ddp()
    
    # 2. Preparar datos con el DataLoader modificado para DDP
    train_dl, test_dl, train_sampler = get_mnist_dataloaders_ddp(
        batch_size=128, 
        num_workers=4 
    )
    
    # 3. Instanciar matemática de difusión y modelo (Canales configurados a 1)
    diffusion_params = GaussianDiffusion(T=1000, device=local_rank)
    unet_model = DDPM_UNet(in_channels=1, out_channels=1).to(local_rank)
    
    # 4. Configurar optimizador y su tasa de aprendizaje decreciente
    optimizer = torch.optim.Adam(unet_model.parameters(), lr=2e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)
    
    # 5. Entrenar distribuidamente
    trained_model = train_diffusion_model_ddp(
        model=unet_model,
        dataloader=train_dl,
        sampler=train_sampler,
        optimizer=optimizer,
        scheduler=scheduler,
        diffusion=diffusion_params,
        epochs=50,
        local_rank=local_rank,
        experiment_name=experiment_name
    )
    
    # 6. Evaluación FID (Solo en la GPU principal para evitar conflictos de memoria)
    if local_rank == 0:
        print("\nIniciando cálculo de FID...")
        score = calculate_fid(
            model=trained_model.module,
            diffusion=diffusion_params,
            dataloader=train_dl,
            device=local_rank,
            num_samples=10000, 
            batch_size=128
        )
        print(f"\n=======================")
        print(f"FID Final: {score:.4f}")
        print(f"=======================\n")
        
        # Guardar los pesos finales
        torch.save(trained_model.module.state_dict(), f"{experiment_name}/ddpm_mnist_final.pth")
        print(f"Pesos del modelo guardados en '{experiment_name}/ddpm_mnist_final.pth'")

    # 7. Limpiar los procesos activos de DDP
    cleanup_ddp()

In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/ddpm_ddp.py